# AI Gimbal Camera - Live Demo
Live camera feed with face detection (FCN), Kalman tracking, gesture recognition, and mock gimbal control. No hardware required.

In [ ]:
import sys, os, time, threading, cv2, numpy as np
from IPython.display import display
import ipywidgets as widgets

sys.path.append('..')

from src.capture.camera import Camera
from src.cv.face_detector_cnn import FaceCNN
from src.cv.face_tracker import KalmanTracker
from src.cv.gesture_classifier import GestureClassifier, HandDetector
from src.control.gimbal import GimbalController
from src.control.pid import PIDController
from src.control.state_machine import StateMachine, Mode
from src.utils.config import load_config
from src.utils.visualization import draw_debug_overlay, compute_framing_error

print('All imports loaded')

In [ ]:
cfg = load_config('../config/default.yaml')
print(f'Config: {cfg.camera.width}x{cfg.camera.height}')

In [ ]:
class LiveDemo:
    def __init__(self, cfg):
        self.cfg = cfg
        self.camera = Camera(
            source=cfg.camera.source, width=cfg.camera.width,
            height=cfg.camera.height, fps=cfg.camera.fps,
            processing_width=cfg.camera.processing_width,
            processing_height=cfg.camera.processing_height,
            use_capture_thread=False)
        self.face_cnn = FaceCNN(
            model_path=cfg.models.face_cnn,
            confidence_threshold=cfg.face_detection.confidence_threshold)
        self.kalman = KalmanTracker(
            max_lost_frames=cfg.kalman.max_lost_frames)
        self.hand = HandDetector(
            ycrcb_lower=tuple(cfg.hand_detection.ycrcb_lower),
            ycrcb_upper=tuple(cfg.hand_detection.ycrcb_upper),
            min_area=cfg.hand_detection.min_area,
            use_motion=False)
        self.gesture = GestureClassifier(
            min_confidence=cfg.gesture.min_confidence)
        self.gimbal = GimbalController(
            port=cfg.serial.port, baud=cfg.serial.baud)
        self.pid_pan = PIDController(
            Kp=cfg.pid.pan.Kp, Ki=cfg.pid.pan.Ki, Kd=cfg.pid.pan.Kd,
            output_limits=tuple(cfg.pid.pan.output_limits),
            integral_limit=cfg.pid.pan.integral_limit)
        self.pid_tilt = PIDController(
            Kp=cfg.pid.tilt.Kp, Ki=cfg.pid.tilt.Ki, Kd=cfg.pid.tilt.Kd,
            output_limits=tuple(cfg.pid.tilt.output_limits),
            integral_limit=cfg.pid.tilt.integral_limit)
        self.sm = StateMachine(idle_timeout_frames=cfg.state_machine.idle_timeout_frames)
        self.running = False
        self.lock = threading.Lock()
        self.frame = None
        self.face = None
        self.gesture_result = None
        self.fps = 0.0

    def start(self):
        self.running = True
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def stop(self):
        self.running = False
        if self.thread and self.thread.is_alive():
            self.thread.join(timeout=2.0)
        self.camera.release()

    def _loop(self):
        fc = 0
        fps_t = time.time()
        fps_c = 0
        last_t = time.time()
        while self.running:
            fo = self.camera.read()
            if fo is None: continue
            frame = fo.data
            fc += 1; fps_c += 1
            now = time.time()
            if now - fps_t >= 1.0:
                self.fps = fps_c / (now - fps_t)
                fps_c = 0; fps_t = now
            h, w = frame.shape[:2]
            dt = now - last_t; last_t = now
            proc = self.camera.get_processing_frame(frame)
            faces = self.face_cnn.detect(proc)
            face = self.kalman.update(faces, dt)
            if face:
                sx = w / self.camera.processing_width
                sy = h / self.camera.processing_height
                for attr in ['x','y','w','h']:
                    setattr(face.bbox, attr, int(getattr(face.bbox, attr) * (sx if attr in ('x','w') else sy)))
            self.sm.update_face_status(face is not None)
            if self.sm.mode == Mode.TRACKING and face:
                ex, ey = compute_framing_error(face.bbox, (w, h), 0.05)
                self.gimbal.set_pan_delta(self.pid_pan.update(ex, dt))
                self.gimbal.set_tilt_delta(self.pid_tilt.update(ey, dt))
            gr = None
            if fc % 6 == 0:
                hr = self.hand.detect(frame, face.bbox if face else None)
                if hr:
                    gr = self.gesture.predict(hr[0])
            ov = draw_debug_overlay(
                frame=frame, face=face, gesture=gr, mode=self.sm.mode,
                fps=self.fps,
                gimbal_angles=(self.gimbal.pan_angle, self.gimbal.tilt_angle),
                kalman_uncertainty=self.kalman.uncertainty)
            with self.lock:
                self.frame = ov; self.face = face; self.gesture_result = gr

    def get_frame(self):
        with self.lock:
            return self.frame.copy() if self.frame is not None else None

In [ ]:
demo = LiveDemo(cfg)
demo.start()
print('Demo started. Stop with the button below.')

In [ ]:
h = widgets.Image()
display(h)
try:
    while True:
        f = demo.get_frame()
        if f is not None:
            _, jpeg = cv2.imencode('.jpg', f, [int(cv2.IMWRITE_JPEG_QUALITY), 70])
            h.value = jpeg.tobytes()
        time.sleep(0.03)
except KeyboardInterrupt:
    pass
finally:
    demo.stop()
    print('Done')

In [ ]:
def home_cb(_): demo.gimbal.home()
def lock_cb(_): demo.sm.toggle_lock()
b_home = widgets.Button(description='Home')
b_lock = widgets.Button(description='Toggle Lock')
b_stop = widgets.Button(description='Stop')
b_home.on_click(home_cb)
b_lock.on_click(lock_cb)
b_stop.on_click(lambda _: setattr(demo, 'running', False))
widgets.VBox([widgets.HBox([b_home, b_lock, b_stop])])